# 03 — Electricity pricing calibration

Per-country traction electricity price in EUR/kWh, written to
`data/electricity_prices.csv`. Consumed by the energy model, which multiplies
it by `energy_kwh` per country leg — this notebook prices the kWh, it does not
model consumption.

Narrative in `ELECTRICITY_PRICING.md`. Three derivation modes:

- `benchmark` — Eurostat band IE components, for the majority of countries
  where no IM sells energy at a published tariff
- `im_tariff` — an IM publishes an all-in traction price (CH, HR, HU, SE), or a
  national statistic exists (UK)
- `network_override` — Eurostat for the commodity, but a dedicated
  traction-network tariff replaces the network component (AT, DE, FR)

In [ ]:
# STDLIB-ONLY cell. See the seed-export contract in calib/README.md.
import csv
from pathlib import Path


def _calib_dir() -> Path:
    """Anchor on the calib folder whatever the kernel's cwd happens to be."""
    cwd = Path.cwd().resolve()
    for cand in (cwd, *cwd.parents):
        if (cand / "resolution.py").exists():
            return cand
    for sub in ("backend/models/infrastructure/calib", "models/infrastructure/calib"):
        if (cwd / sub).is_dir():
            return (cwd / sub).resolve()
    raise RuntimeError("cannot locate models/infrastructure/calib")


CALIB_DIR = _calib_dir()
DATA_DIR = CALIB_DIR / "electricity_pricing" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)


def write_data(name: str, fieldnames: list[str], rows: list[dict]) -> None:
    """Write one committed observation table to calib/electricity_pricing/data/."""
    with open(DATA_DIR / name, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        w.writerows(rows)
    print(f"electricity_pricing/data/{name}: {len(rows)} rows")


def read_data(name: str) -> list[dict]:
    with open(DATA_DIR / name, newline="", encoding="utf-8") as fh:
        return list(csv.DictReader(fh))

In [ ]:
# STDLIB-ONLY cell.
from dataclasses import dataclass, asdict
from typing import Optional

# How much evidence stands behind a value. Not a quality judgement — a
# well-argued ASSUMED and a mis-transcribed SOURCED are both possible; the
# status says which kind of thing the reader is looking at.
SOURCED = "sourced"  # named document, named locator
NOT_LEVIED = "not_levied"  # positively documented as zero — not absent data
DERIVED = "derived"  # arithmetic on other values, formula in the note
BENCHMARK = "benchmark"  # pan-European statistic standing in for a country
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
MISSING = "missing"  # nothing read yet — never an estimate
NO_RAILWAY = "no_railway"

USABLE = {SOURCED, NOT_LEVIED, DERIVED, BENCHMARK, ASSUMED}


@dataclass(frozen=True)
class SV:
    """
    One calibrated parameter with its audit trail.

    source_id points at sources_register.csv; locator is what makes it
    re-checkable a year later (section, table, sheet), because a network
    statement runs to hundreds of pages and its tariff tables move between
    editions. Values stay in native currency and price basis — conversion and
    escalation happen later and explicitly.
    """

    country_code: str
    parameter: str
    value: Optional[float]
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: Optional[int] = None
    note: str = ""
    low: Optional[float] = None  # sensitivity band, mandatory when ASSUMED
    high: Optional[float] = None

    def __post_init__(self):
        if self.status in USABLE and self.value is None:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: {self.status} needs a value"
            )
        if self.status in (SOURCED, NOT_LEVIED) and not (
            self.source_id and self.locator
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: sourced needs source_id + locator"
            )
        if self.status == ASSUMED and (
            self.low is None or self.high is None or not self.note
        ):
            raise ValueError(
                f"{self.country_code}/{self.parameter}: assumed needs a band and a rationale"
            )
        if self.status == DERIVED and not self.note:
            raise ValueError(
                f"{self.country_code}/{self.parameter}: derived must state its formula"
            )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]


def emit(name: str, values: list[SV]) -> None:
    write_data(name, SV_FIELDS, [asdict(v) for v in values])
    by_status: dict[str, int] = {}
    for v in values:
        by_status[v.status] = by_status.get(v.status, 0) + 1
    for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
        print(f"    {s:11} {n:4}")

## Eurostat band IE components

Band IE is 20,000–69,999 MWh/a, where a night-train operation lands at
15–25 kWh/train-km over 1–3 Mio train-km a year. The choice is validated rather
than asserted: Hungary's published all-in traction tariff and Sweden's
Trafikverket example both reproduce their band IE totals almost exactly, which
they would not at any other band.

In [ ]:
LOC = "band IE 2025"
COMPONENTS = (
    "energy_supply",
    "network",
    "vat",
    "renewable",
    "capacity",
    "environmental",
    "nuclear",
    "other",
)
NON_VAT = [c for c in COMPONENTS if c != "vat"]

EUROSTAT_IE = {
    "AT": (0.1112, 0.0292, 0.0322, 0.0047, 0.0000, 0.0135, 0.0000, 0.0024),
    "BE": (0.0968, 0.0169, 0.0279, 0.0113, 0.0008, 0.0072, 0.0000, 0.0000),
    "BG": (0.1108, 0.0219, 0.0251, 0.0000, 0.0000, 0.0010, 0.0000, -0.0082),
    "CZ": (0.1114, 0.0414, 0.0351, 0.0129, 0.0000, 0.0012, 0.0000, 0.0001),
    "DE": (0.0979, 0.0375, 0.0321, 0.0017, 0.0097, 0.0205, 0.0000, 0.0014),
    "DK": (0.0912, 0.0261, 0.0534, 0.0000, 0.0000, 0.0011, 0.0000, 0.0000),
    "EE": (0.0811, 0.0209, 0.0269, 0.0084, 0.0000, 0.0018, 0.0000, 0.0000),
    "ES": (0.0896, 0.0100, 0.0222, 0.0007, 0.0002, 0.0035, 0.0000, 0.0050),
    "FI": (0.0375, 0.0134, 0.0131, 0.0000, 0.0001, 0.0005, 0.0000, 0.0000),
    "FR": (0.0680, 0.0175, 0.0141, 0.0000, 0.0012, 0.0057, 0.0000, 0.0000),
    "GR": (0.1344, 0.0112, 0.0093, 0.0028, 0.0000, 0.0020, 0.0000, 0.0054),
    "HR": (0.1034, 0.0171, 0.0172, 0.0111, 0.0000, 0.0005, 0.0000, 0.0000),
    "HU": (0.1175, 0.0352, 0.0415, 0.0135, 0.0000, 0.0010, 0.0000, 0.0039),
    "IE": (0.1529, 0.0474, 0.0129, 0.0019, 0.0001, 0.0003, 0.0000, 0.0040),
    "IT": (0.1223, 0.0166, 0.0158, 0.0147, 0.0083, 0.0015, 0.0000, 0.0016),
    "LT": (0.0940, 0.0258, 0.0250, 0.0002, 0.0000, 0.0000, 0.0000, 0.0002),
    "LU": (0.1036, 0.0110, 0.0092, 0.0003, 0.0000, 0.0000, 0.0000, 0.0002),
    "LV": (0.0871, 0.0111, 0.0214, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039),
    "NL": (0.0959, 0.0200, 0.0271, 0.0000, 0.0000, 0.0132, 0.0000, 0.0000),
    "NO": (0.0516, 0.0053, 0.0163, 0.0000, 0.0000, 0.0082, 0.0000, 0.0000),
    "PL": (0.0775, 0.0278, 0.0373, 0.0022, 0.0099, 0.0446, 0.0000, 0.0003),
    "PT": (0.0814, 0.0146, 0.0235, 0.0079, 0.0005, 0.0005, 0.0000, 0.0011),
    "RO": (0.1128, 0.0311, 0.0313, 0.0128, 0.0000, 0.0005, 0.0000, 0.0000),
    "SE": (0.0530, 0.0161, 0.0174, 0.0000, 0.0000, 0.0005, 0.0000, 0.0000),
    "SI": (0.1130, 0.0126, 0.0290, 0.0050, 0.0001, 0.0009, 0.0000, 0.0000),
    "SK": (0.1135, 0.0344, 0.0329, 0.0111, 0.0096, 0.0013, 0.0033, 0.0000),
}
print(f"{len(EUROSTAT_IE)} countries on the Eurostat basis")

## Rail-specific electricity tax

Eurostat's tax columns are generic non-household figures. Whether *rail
traction* is exempt or reduced under Art. 15(1)(e) of Directive 2003/96/EC is a
separate fact, and the rail rate **replaces** Eurostat's environmental
component rather than adding to it — they are the same tax at different rates.
Germany proves the identity: Eurostat reads 0.0205 EUR/kWh, exactly the
standard Stromsteuer rate of 20.50 EUR/MWh, against a rail rate of 11.42.

In [ ]:
_CD, _CDL = "CE-DELFT-4K83", "Rail_Energy taxes_level"

RAIL_TAX: dict[str, SV] = {}
for cc, v in {
    "BG": 0.002145,
    "DK": 0.000372,
    "EE": 0.006102,
    "ES": 0.005677,
    "FR": 0.000457,
    "GR": 0.000609,
    "HU": 0.001691,
    "LT": 0.000848,
    "LU": 0.000414,
    "NL": 0.002332,
    "PL": 0.008234,
    "RO": 0.001039,
    "SI": 0.003968,
}.items():
    RAIL_TAX[cc] = SV(
        cc,
        "rail_electricity_tax",
        v,
        "EUR/kWh",
        SOURCED,
        _CD,
        _CDL,
        "EUR",
        2016,
        "PPS-adjusted; nominal differs by the country price level",
    )

for cc in (
    "BE",
    "CZ",
    "FI",
    "HR",
    "IE",
    "IT",
    "LV",
    "NO",
    "PT",
    "SE",
    "SK",
    "UK",
    "CH",
):
    RAIL_TAX[cc] = SV(
        cc,
        "rail_electricity_tax",
        0.0,
        "EUR/kWh",
        NOT_LEVIED,
        _CD,
        _CDL,
        "EUR",
        2016,
        "rail traction exempt or not levied",
    )

# The two rates large enough to matter are both known at nominal and both
# confirmed twice, which is why they use nominal rather than PPS values.
RAIL_TAX["AT"] = SV(
    "AT",
    "rail_electricity_tax",
    0.0150,
    "EUR/kWh",
    SOURCED,
    "APS-STROMSTEUER",
    "EU comparison chart",
    "EUR",
    2016,
    "Elektrizitaetsabgabe full rate, no rail carve-out; the 2022-2024 reduction to "
    "1 EUR/MWh expired 31 Dec 2024. Corroborated by CE-DELFT-4K83 at 13.79 PPS.",
)
RAIL_TAX["DE"] = SV(
    "DE",
    "rail_electricity_tax",
    0.01142,
    "EUR/kWh",
    SOURCED,
    "DE-APS-2027",
    "§2.3.3.3 footnote",
    "EUR",
    2027,
    "StromStG §9 reduced rail rate on presentation of an Erlaubnisschein; identical "
    "to the 2016 CE Delft figure, so the rate has not moved in a decade.",
)
print(f"{len(RAIL_TAX)} rail tax entries")

## Overrides and VAT

Germany's traction-network tariff is two-part, so the per-kWh equivalent
depends on Benutzungsdauer — annual running hours over the fleet's
peak-to-average power ratio. Night-only operation lands at 1,750–2,060 h/a,
which fixes the band; inside it the spread is only ±4 %.

VAT is a real cost only where the operator's own supply is VAT-**exempt**,
since exemption removes the deduction right. Denmark is the confirmed
exception. This is the single assumption that moves a country's price by
15–30 %, so it is a per-country flag rather than a constant.

In [ ]:
FX = {"CHF": 1.064, "HUF": 1 / 396.0, "SEK": 1 / 11.2, "GBP": 1.20}

IM_TARIFF = {
    "CH": SV(
        "CH",
        "im_tariff",
        0.078,
        "CHF/kWh",
        SOURCED,
        "CH-NZV",
        "Art.20a transitional",
        "CHF",
        2027,
        "22:00-06:00 rate, -40% on the 0.13 base; all-in, no separate excise",
    ),
    "HR": SV(
        "HR",
        "im_tariff",
        0.0958,
        "EUR/kWh",
        DERIVED,
        "HR-NS-2027",
        "ch.5.4 item 40",
        "EUR",
        2027,
        "night tariff 0.0826 + renewables levy 0.0132; excise printed as 0.000000",
    ),
    "HU": SV(
        "HU",
        "im_tariff",
        67.4,
        "HUF/kWh",
        SOURCED,
        "HU-NS-2627",
        "Annex 5.2-6",
        "HUF",
        2027,
        "49.0 energy + 10.2 system + 0.4 excise + 7.8 funds, all-in",
    ),
    "SE": SV(
        "SE",
        "im_tariff",
        0.7156,
        "SEK/kWh",
        DERIVED,
        "SE-NS-2027",
        "§7.3.11",
        "SEK",
        2027,
        "energy 0.6075 + grid 0.1081",
    ),
    "UK": SV(
        "UK",
        "im_tariff",
        0.238480,
        "GBP/kWh",
        SOURCED,
        "DESNZ-T341",
        "Table 3.4.1 Large band",
        "GBP",
        2025,
        "excl. CCL — rail traction is CCL-exempt, so that is the working series",
    ),
}

NETWORK_OVERRIDE = {
    "AT": SV(
        "AT",
        "network_override",
        0.04367,
        "EUR/kWh",
        SOURCED,
        "AT-SNNB-2026",
        "Tab.59 p.114",
        "EUR",
        2026,
        "Bahnstromnetz Niedertarif 22:00-06:00 (43.67 EUR/MWh); Hochtarif 52.40",
    ),
    "DE": SV(
        "DE",
        "network_override",
        0.0844,
        "EUR/kWh",
        ASSUMED,
        "DE-DBE-NETZ-2026",
        "§1",
        "EUR",
        2026,
        "Arbeitspreis 0.0721 (Hochspannung <2500 h/a) + Leistungspreis 23.35 EUR/kWa at an "
        "assumed 1,900 h/a Benutzungsdauer. Night-only operation cannot reach the >=2500 h/a "
        "band, which needs ~4,250 running hours a year. Statutory levies are not added: they "
        "already sit in Eurostat's tax columns, which this override leaves untouched.",
        low=0.0818,
        high=0.0877,
    ),
    "FR": SV(
        "FR",
        "network_override",
        0.0315,
        "EUR/kWh",
        DERIVED,
        "FR-DRR-A512",
        "§2.1",
        "EUR",
        2027,
        "RCTE-A = purchase price 0.0680 x loss rate 0.134/(1-0.134) = 0.0105, plus RCTE-B "
        "0.0210 indexed from 0.02003. Formula validated by back-solving the 2024 tariff to a "
        "0.1850 EUR/kWh purchase price, matching the published 2024 RFE of 0.18653.",
    ),
}

VAT_RECOVERABLE = {cc: True for cc in set(EUROSTAT_IE) | set(IM_TARIFF)}
VAT_RECOVERABLE["DK"] = False
VAT_FLAG = SV(
    "DK",
    "vat_non_deductible",
    1.0,
    "flag",
    SOURCED,
    "SKAT-DK-VAT",
    "Passenger transport section",
    "EUR",
    2024,
    "passenger transport is VAT-exempt, so input VAT is not deductible",
)

# Poland is the one documented departure from the replacement rule: its
# environmental component is ~37x the Polish excise, so it plainly bundles
# other levies and subtracting it would over-credit. Erring high.
PL_NO_RECONCILE = True
print(f"non-recoverable VAT: {[cc for cc, ok in VAT_RECOVERABLE.items() if not ok]}")

In [ ]:
VALUES: list[SV] = []
MODE_OUT: list[dict] = []

for cc in sorted(set(EUROSTAT_IE) | set(IM_TARIFF)):
    if cc in IM_TARIFF:
        t = IM_TARIFF[cc]
        price = t.value * FX.get(t.currency, 1.0)
        VALUES.append(t)
        VALUES.append(
            SV(
                cc,
                "working_price",
                round(price, 4),
                "EUR/kWh",
                DERIVED,
                t.source_id,
                t.locator,
                "EUR",
                t.basis_year,
                f"{t.value:g} {t.currency} converted at {FX.get(t.currency, 1.0):g} EUR/{t.currency} (ECB-FX)",
            )
        )
        mode = "im_tariff"
    else:
        comp = dict(zip(COMPONENTS, EUROSTAT_IE[cc]))
        for name in COMPONENTS:
            VALUES.append(
                SV(
                    cc,
                    f"eurostat_{name}",
                    comp[name],
                    "EUR/kWh",
                    BENCHMARK,
                    "EUROSTAT-NRG-PC-205-C",
                    LOC,
                    "EUR",
                    2025,
                    "price component, band IE",
                )
            )
        total = sum(comp[c] for c in NON_VAT)
        mode = "benchmark"

        if cc in NETWORK_OVERRIDE:
            ov = NETWORK_OVERRIDE[cc]
            VALUES.append(ov)
            total = total - comp["network"] + ov.value
            mode = "network_override"

        tax = RAIL_TAX.get(cc)
        if tax is not None:
            VALUES.append(tax)
            if not (cc == "PL" and PL_NO_RECONCILE):
                total = total - comp["environmental"] + tax.value

        if not VAT_RECOVERABLE[cc]:
            total += comp["vat"]
            VALUES.append(VAT_FLAG)

        price = total
        VALUES.append(
            SV(
                cc,
                "working_price",
                round(price, 4),
                "EUR/kWh",
                DERIVED,
                "EUROSTAT-NRG-PC-205-C",
                LOC,
                "EUR",
                2025,
                "sum(non-VAT components) - environmental + rail tax"
                + (" + non-deductible VAT" if not VAT_RECOVERABLE[cc] else "")
                + (
                    " [PL: no reconciliation, environmental column bundles non-excise levies]"
                    if cc == "PL" and PL_NO_RECONCILE
                    else ""
                ),
            )
        )

    MODE_OUT.append(
        dict(
            country_code=cc,
            derivation_mode=mode,
            working_price_eur_kwh=round(price, 4),
            vat_recoverable=VAT_RECOVERABLE[cc],
        )
    )

assert all(0.03 <= r["working_price_eur_kwh"] <= 0.40 for r in MODE_OUT), (
    "price outside plausible band"
)
write_data(
    "electricity_price_modes.csv",
    ["country_code", "derivation_mode", "working_price_eur_kwh", "vat_recoverable"],
    MODE_OUT,
)
emit("electricity_prices.csv", VALUES)

In [ ]:
# Display / validation only — pandas is fine here, seed.py skips this cell.
import pandas as pd

_df = pd.DataFrame([asdict(v) for v in VALUES])
_reg = set(pd.read_csv(DATA_DIR / "sources_register.csv")["source_id"])
_unknown = set(_df.loc[_df.source_id.ne(""), "source_id"]) - _reg
assert not _unknown, f"unregistered source ids: {sorted(_unknown)}"
_bad = _df[(_df.status == "assumed") & (_df.low.isna() | _df.high.isna())]
assert _bad.empty, _bad
print(f"{len(_df)} values, {_df.source_id.ne('').sum()} with a source pointer")
pd.DataFrame(MODE_OUT).sort_values("working_price_eur_kwh")